# Job Market Intelligence – Data Collection

This notebook runs the No Fluff Jobs data-collection pipeline.

Scraping logic is implemented in the `src/nfj` package. Raw datasets are stored in `data/raw/` and excluded from version control. All network operations are disabled by default and must be enabled explicitly with the corresponding `RUN_*` flag.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from nfj import (
    collect_job_urls,
    refresh_job_records,
    refresh_missing_salary_periods,
    scrape_jobs,
)
from nfj.config import URLS_PATH

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
JOBS_PATH = RAW_DATA_DIR / "nofluff_it_jobs_refresh.csv"
ERRORS_PATH = RAW_DATA_DIR / "nofluff_scraping_errors_refresh.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("URL file:", URLS_PATH)
print("Job file:", JOBS_PATH)


## 1. Collect job URLs

Collect the current set of unique job-offer URLs from all configured No Fluff Jobs categories.


In [ ]:
RUN_URL_COLLECTION = False

if RUN_URL_COLLECTION:
    df_urls = collect_job_urls()
else:
    print("URL collection skipped.")


## 2. Scrape job offers

Scrape offers from the URL list and save them to the active raw dataset. Existing records are retained, while URLs that have not been collected are scraped.


In [ ]:
RUN_SCRAPING = False

if RUN_SCRAPING:
    df_jobs = scrape_jobs(
        urls_path=URLS_PATH,
        output_path=JOBS_PATH,
        error_path=ERRORS_PATH,
        delay=3.0,
        headless=False,
        max_consecutive_page_errors=5,
    )
else:
    print("Job scraping skipped.")


## 3. Refresh missing salary periods

Re-scrape paid offers whose `salary_period` is missing. Existing salary values are preserved when the period still cannot be identified.


In [ ]:
RUN_SALARY_PERIOD_REFRESH = False

if RUN_SALARY_PERIOD_REFRESH:
    df_jobs = refresh_missing_salary_periods(
        output_path=JOBS_PATH,
        error_path=ERRORS_PATH,
        delay=3.0,
        headless=False,
    )
else:
    print("Salary-period refresh skipped.")


## 4. Refresh selected offers

Use this section only when specific records need to be scraped again after a parser correction. Add exact URLs to `URLS_TO_REFRESH` and enable the flag.


In [ ]:
RUN_SELECTED_REFRESH = False

URLS_TO_REFRESH = []

if RUN_SELECTED_REFRESH:
    if not URLS_TO_REFRESH:
        raise ValueError(
            "Add at least one URL to URLS_TO_REFRESH before enabling the refresh."
        )

    df_jobs = refresh_job_records(
        urls=URLS_TO_REFRESH,
        output_path=JOBS_PATH,
        error_path=ERRORS_PATH,
        delay=3.0,
        headless=False,
    )
else:
    print("Selected-offer refresh skipped.")


## 5. Validate collected data

Check completeness, URL uniqueness, invalid pages, and fields affected by the parser corrections.


In [ ]:
df_urls = pd.read_csv(URLS_PATH)
df_jobs = pd.read_csv(JOBS_PATH)

invalid_page_markers = (
    "Ta strona nie działa|"
    "Oferta pracy wygasła|"
    "Oferta wygasła"
)

invalid_job_mask = (
    df_jobs["title"]
    .fillna("")
    .str.contains(
        invalid_page_markers,
        case=False,
        regex=True,
    )
)

required_skills_leak = (
    df_jobs["required_skills"]
    .fillna("")
    .str.contains("Opis oferty", case=False, regex=False)
)

responsibilities_leak = (
    df_jobs["responsibilities"]
    .fillna("")
    .str.contains("Opis oferty", case=False, regex=False)
)

salary_without_period = (
    df_jobs[["salary_min", "salary_max", "salary_currency"]]
    .notna()
    .any(axis=1)
    & df_jobs["salary_period"].isna()
)

missing_urls = set(df_urls["url"]) - set(df_jobs["url"])

print("Collected URLs:", len(df_urls))
print("Unique collected URLs:", df_urls["url"].nunique())
print()
print("Job records:", len(df_jobs))
print("Unique job URLs:", df_jobs["url"].nunique())
print("Duplicate job URLs:", df_jobs["url"].duplicated().sum())
print("URLs without a job record:", len(missing_urls))
print()
print("Invalid job pages:", int(invalid_job_mask.sum()))
print("Salary values without period:", int(salary_without_period.sum()))
print("Required-skills section leaks:", int(required_skills_leak.sum()))
print("Responsibilities section leaks:", int(responsibilities_leak.sum()))
print("Missing categories:", int(df_jobs["category"].isna().sum()))
